# Legal Case Retrieval Agent — Kaggle T4 测试

在 Kaggle T4×2 上跑通完整流程：
1. 安装依赖
2. 下载项目代码
3. 下载 BGE-M3 + LawLLM-7B 模型
4. 下载并处理 CAIL2018 数据
5. 构建向量索引
6. 跑通完整 pipeline

**注意**：请在 Kaggle Notebook 设置中选择 **T4×2 GPU** 和 **Internet ON**

## Cell 1: 安装依赖

In [ ]:
# 安装依赖（Kaggle 已预装 torch，不需要重装）
!pip install -q vllm==0.6.3
!pip install -q FlagEmbedding sentence-transformers
!pip install -q qdrant-client jieba rank-bm25
!pip install -q datasets pyyaml loguru

# 验证 GPU
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
print(f'GPU count: {torch.cuda.device_count()}')
for i in range(torch.cuda.device_count()):
    print(f'  GPU {i}: {torch.cuda.get_device_name(i)} ({torch.cuda.get_device_properties(i).total_memory / 1024**3:.1f} GB)')

## Cell 2: 下载项目代码

把你的项目代码放到 Kaggle 上。有两种方式：
- 方式 A：如果代码在 GitHub 上，直接 git clone
- 方式 B：把项目打成 zip 上传到 Kaggle Dataset，然后挂载

以下假设用 git clone（如果用 Dataset 方式，跳过此 cell，改用对应路径）

In [ ]:
# ===== 方式 A: git clone =====
# 替换成你的仓库地址（如果私有仓库需要 token）
# !git clone https://github.com/your-username/Legal-Case-Retrieval-Agent.git /kaggle/working/Legal-Case-Retrieval-Agent

# ===== 方式 B: 直接在 Kaggle 上创建项目结构 =====
# 如果没有 GitHub 仓库，可以直接把代码文件创建在这里
# 下面我们直接在 /kaggle/working/ 下重建项目

import os
PROJECT_DIR = '/kaggle/working/Legal-Case-Retrieval-Agent'
os.makedirs(PROJECT_DIR, exist_ok=True)
os.chdir(PROJECT_DIR)
print(f'工作目录: {os.getcwd()}')

# 如果你是通过 Kaggle Dataset 上传的项目代码，取消下面的注释并修改路径：
# !cp -r /kaggle/input/your-dataset-name/* {PROJECT_DIR}/

## Cell 3: 下载 BGE-M3 模型

BGE-M3 约 2.3GB，T4 上用 FP16 跑 embedding 没问题。

In [ ]:
import os
os.environ['HF_ENDPOINT'] = 'https://hf-mirror.com'  # 国内镜像加速

from huggingface_hub import snapshot_download

# 下载 BGE-M3 embedding 模型
bge_path = '/kaggle/working/models/bge-m3'
print('下载 BGE-M3...')
snapshot_download(
    'BAAI/bge-m3',
    local_dir=bge_path,
    repo_type='model',
)
print(f'BGE-M3 下载完成: {bge_path}')

## Cell 4: 下载 LawLLM-7B 模型

LawLLM-7B 约 15GB。T4 16GB 显存需要用 AWQ 量化版本（约 5-6GB）。

**注意**：如果 LawLLM-7B 没有 AWQ 量化版本，可以用 FP16 原始版本，
vLLM 会自动管理显存。T4×2 共 32GB 够用。

In [ ]:
# 下载 LawLLM-7B
llm_path = '/kaggle/working/models/LawLLM-7B'
print('下载 LawLLM-7B...（约 15GB，需要几分钟）')
snapshot_download(
    'ShengbinYue/LawLLM-7B',
    local_dir=llm_path,
    repo_type='model',
)
print(f'LawLLM-7B 下载完成: {llm_path}')

# 检查模型大小
import subprocess
result = subprocess.run(['du', '-sh', llm_path], capture_output=True, text=True)
print(result.stdout)

## Cell 5: 下载并处理 CAIL2018 数据

In [ ]:
import os, sys, json
os.chdir(PROJECT_DIR)
sys.path.insert(0, PROJECT_DIR)

# 确保项目目录结构存在
os.makedirs('data/raw', exist_ok=True)
os.makedirs('data/processed', exist_ok=True)
os.makedirs('configs', exist_ok=True)

# 写入 config.yaml（使用本地模型路径）
config_yaml = """model:
  llm_path: "/kaggle/working/models/LawLLM-7B"
  embedding_path: "/kaggle/working/models/bge-m3"
  reranker_path: "BAAI/bge-reranker-v2-m3"

api:
  base_url: ""
  api_key: "EMPTY"
  model_name: "ShengbinYue/LawLLM-7B"

vector_store:
  type: "qdrant"
  mode: "embedded"
  path: "qdrant_data"
  host: "localhost"
  port: 6333
  collection_name: "legal_cases"
  vector_dim: 1024

retrieval:
  top_k_vector: 50
  top_k_keyword: 50
  top_k_final: 5

rerank:
  weights:
    cause_of_action: 40
    focus_similarity: 30
    fact_overlap: 20
    same_region: 5
    recency: 5
"""
with open('configs/config.yaml', 'w', encoding='utf-8') as f:
    f.write(config_yaml)
print('config.yaml 已写入')

# 下载 CAIL2018 数据（取前 1000 条测试）
from datasets import load_dataset
print('下载 CAIL2018 数据...')
ds = load_dataset('china-ai-law-challenge/cail2018')

# 取第一个 split 的前 1000 条
first_split = list(ds.keys())[0]
raw_path = 'data/raw/cail2018/cail2018.jsonl'
os.makedirs('data/raw/cail2018', exist_ok=True)
with open(raw_path, 'w', encoding='utf-8') as f:
    count = 0
    for row in ds[first_split]:
        if count >= 1000:
            break
        row_dict = {k: v for k, v in row.items()}
        f.write(json.dumps(row_dict, ensure_ascii=False) + '\n')
        count += 1
print(f'下载了 {count} 条 CAIL2018 数据')

## Cell 6: 处理数据（解析 → 去重 → 校验）

需要项目代码文件存在。如果你的代码是通过 Kaggle Dataset 上传的，
确保 src/ 目录在 PROJECT_DIR 下。

如果 src/ 目录不存在，需要先创建项目代码文件（见下方说明）。

In [ ]:
import os, sys, json
os.chdir(PROJECT_DIR)
sys.path.insert(0, PROJECT_DIR)

# 检查 src 目录是否存在
if not os.path.exists('src/__init__.py'):
    print('⚠️ src/ 目录不存在！请确保项目代码已上传到 Kaggle。')
    print('你可以通过以下方式上传：')
    print('  1. 把项目打包成 zip，上传为 Kaggle Dataset')
    print('  2. 在 Kaggle Notebook 右侧 Add Input → 选你的 Dataset')
    print('  3. 然后把文件复制到项目目录')
    print()
    print('或者如果你有 GitHub 仓库，在 Cell 2 用 git clone')
else:
    # 处理数据
    from src.data_processing.case_parser import parse_record, deduplicate, validate
    
    # 加载原始数据
    records = []
    with open('data/raw/cail2018/cail2018.jsonl', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    print(f'加载了 {len(records)} 条原始记录')
    
    # 解析
    cases = []
    for i, record in enumerate(records):
        try:
            case = parse_record('cail2018', record, index=i)
            cases.append(case)
        except Exception as e:
            pass
    
    # 去重 + 校验
    cases = deduplicate(cases)
    cases = [c for c in cases if validate(c)]
    
    # 输出
    with open('data/processed/cail2018.jsonl', 'w', encoding='utf-8') as f:
        for case in cases:
            f.write(json.dumps(case, ensure_ascii=False) + '\n')
    
    print(f'处理完成: {len(cases)} 条有效案例')
    # 打印样例
    print(f'\n样例:')
    print(f'  case_id: {cases[0]["case_id"]}')
    print(f'  案由: {cases[0]["cause_of_action"]}')
    print(f'  事实: {cases[0]["facts"][:100]}...')

## Cell 7: 用 BGE-M3 构建向量索引

In [ ]:
import os, sys, json
os.chdir(PROJECT_DIR)
sys.path.insert(0, PROJECT_DIR)

# 加载处理后的案例
cases = []
with open('data/processed/cail2018.jsonl', encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        if line:
            cases.append(json.loads(line))
print(f'加载了 {len(cases)} 条案例')

# 加载 BGE-M3
from FlagEmbedding import BGEM3FlagModel
embed_model = BGEM3FlagModel('/kaggle/working/models/bge-m3', use_fp16=True)
print('BGE-M3 已加载')

# 构造 embedding 文本
texts = []
for case in cases:
    cause = case.get('cause_of_action', '')
    facts = case.get('facts', '')
    text = f'案由：{cause}\n事实：{facts}' if cause else facts
    texts.append(text)

# 批量向量化
print(f'开始向量化 {len(texts)} 条文本...')
embeddings = embed_model.encode(texts, batch_size=32, max_length=8192)['dense_vecs']
print(f'向量化完成: shape={embeddings.shape}')

# 写入 Qdrant 嵌入式模式
from qdrant_client import QdrantClient
from qdrant_client.http.models import Distance, PointStruct, VectorParams

# 删除旧数据（如果有）
import shutil
if os.path.exists('qdrant_data'):
    shutil.rmtree('qdrant_data')

client = QdrantClient(path='qdrant_data')
vector_dim = embeddings.shape[1]
client.recreate_collection(
    collection_name='legal_cases',
    vectors_config=VectorParams(size=vector_dim, distance=Distance.COSINE),
)
print(f'已创建 Qdrant collection (dim={vector_dim})')

# 批量写入
batch_size = 100
for i in range(0, len(cases), batch_size):
    batch_cases = cases[i:i+batch_size]
    batch_embs = embeddings[i:i+batch_size].tolist()
    points = [
        PointStruct(id=i+j, vector=emb, payload={
            'case_id': case.get('case_id', ''),
            'case_type': case.get('case_type', ''),
            'cause_of_action': case.get('cause_of_action', ''),
            'facts': case.get('facts', ''),
            'legal_basis': case.get('legal_basis', []),
            'ruling': case.get('ruling', ''),
        })
        for j, (case, emb) in enumerate(zip(batch_cases, batch_embs))
    ]
    client.upsert(collection_name='legal_cases', points=points)
print(f'向量索引构建完成! 共 {len(cases)} 条')

# 验证检索
query_vec = embeddings[0].tolist()
results = client.query_points(collection_name='legal_cases', query=query_vec, limit=3)
print('\n验证检索（用第一条案例做查询）:')
for point in results.points:
    print(f'  score={point.score:.4f} case_id={point.payload.get("case_id")} cause={point.payload.get("cause_of_action")}')

# 释放 embedding 模型显存（给 LLM 腾空间）
del embed_model
import torch
torch.cuda.empty_cache()
print('已释放 embedding 模型显存')

## Cell 8: 加载 LawLLM-7B 并测试推理

In [ ]:
import os, sys
os.chdir(PROJECT_DIR)
sys.path.insert(0, PROJECT_DIR)

from transformers import AutoTokenizer
from vllm import LLM, SamplingParams

model_path = '/kaggle/working/models/LawLLM-7B'
print(f'加载 LawLLM-7B: {model_path}')

tokenizer = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True)

# T4 16GB 需要量化；如果有 AWQ 版本用 quantization='awq'
# 如果没有量化版本，用 fp16 也可以（T4×2 共 32GB 够用）
try:
    llm = LLM(
        model=model_path,
        quantization='awq',
        max_model_len=4096,
        gpu_memory_utilization=0.85,
        trust_remote_code=True,
    )
    print('已加载 AWQ 量化版本')
except Exception as e:
    print(f'AWQ 加载失败: {e}')
    print('尝试 FP16 原始版本...')
    llm = LLM(
        model=model_path,
        max_model_len=4096,
        gpu_memory_utilization=0.85,
        trust_remote_code=True,
        dtype='float16',
    )
    print('已加载 FP16 版本')

# 测试推理
test_prompt = '你是一位法律专家。请简要回答：什么是盗窃罪？'
sampling = SamplingParams(max_tokens=256, temperature=0.3, top_p=0.9)
outputs = llm.generate([test_prompt], sampling)
print('\n=== 测试推理 ===')
print(f'Prompt: {test_prompt}')
print(f'Response: {outputs[0].outputs[0].text}')

## Cell 9: 跑通完整 Pipeline

这里我们不用 pipeline.py 的 LLMInference 类（它通过 vLLM API 调用），
而是直接用上面加载的 vLLM 模型对象，避免重复加载。

In [ ]:
import os, sys, json
os.chdir(PROJECT_DIR)
sys.path.insert(0, PROJECT_DIR)

from src.llm.prompts import format_extract_elements_prompt, format_case_analysis_prompt
from src.retrieval.search import MultiRouteSearcher
from src.retrieval.rerank import Reranker
from vllm import SamplingParams

# 构建检索器（BM25 索引 + Qdrant 向量库）
searcher = MultiRouteSearcher()
searcher.build_bm25_index()  # 从 data/processed/ 加载

# reranker（只用要素加权，不用 BGE-reranker 以节省显存）
reranker = Reranker()

# 封装一个简单的本地推理函数
def llm_generate(prompt, max_tokens=2048, temperature=0.3):
    sampling = SamplingParams(max_tokens=max_tokens, temperature=temperature, top_p=0.9)
    outputs = llm.generate([prompt], sampling)
    return outputs[0].outputs[0].text

def run_pipeline(case_description):
    print('=' * 60)
    print(f'案情: {case_description}')
    print('=' * 60)
    
    # Stage 1: 要素抽取
    print('\n--- Stage 1: 要素抽取 ---')
    prompt = format_extract_elements_prompt(case_description)
    response = llm_generate(prompt, temperature=0.1)
    
    # 解析 JSON
    elements = {}
    try:
        json_str = response
        if '```json' in response:
            json_str = response.split('```json')[1].split('```')[0]
        elif '```' in response:
            json_str = response.split('```')[1].split('```')[0]
        elements = json.loads(json_str.strip())
    except (json.JSONDecodeError, IndexError) as e:
        print(f'JSON 解析失败: {e}')
        elements = {'raw_response': response}
    
    print(f'要素: {json.dumps(elements, ensure_ascii=False, indent=2)}')
    
    # Stage 2: 多路检索
    print('\n--- Stage 2: 多路检索 ---')
    # 向量检索
    from src.retrieval.embed import Embedder
    embedder = Embedder()
    # 重新加载 BGE-M3（因为之前释放了）
    from FlagEmbedding import BGEM3FlagModel
    embedder._model = BGEM3FlagModel('/kaggle/working/models/bge-m3', use_fp16=True)
    
    query_vector = embedder.embed_single(case_description)
    vec_results = searcher.vector_store.search(query_vector, top_k=50)
    print(f'向量检索: {len(vec_results)} 条')
    
    # BM25 检索
    kw_results = searcher.keyword_search(case_description, top_k=50)
    print(f'BM25 检索: {len(kw_results)} 条')
    
    # 合并去重
    seen = set()
    merged = []
    for result in vec_results + kw_results:
        case_id = result.get('payload', {}).get('case_id', '')
        if case_id and case_id not in seen:
            seen.add(case_id)
            merged.append(result)
    print(f'合并去重: {len(merged)} 条')
    
    # Stage 3: 重排序（只用要素加权，跳过 BGE-reranker）
    print('\n--- Stage 3: 重排序 ---')
    # 手动计算要素加权分数（不调用 BGE-reranker）
    for candidate in merged:
        element_score = reranker._element_weighted_score(
            elements, candidate.get('payload', {}), query_text=case_description
        )
        candidate['rerank_score'] = candidate.get('score', 0) + element_score
    
    merged.sort(key=lambda x: x['rerank_score'], reverse=True)
    top_results = merged[:5]
    print(f'Top 5:')
    for i, r in enumerate(top_results):
        payload = r.get('payload', {})
        print(f'  {i+1}. [{r["rerank_score"]:.2f}] {payload.get("cause_of_action", "")} - {payload.get("facts", "")[:50]}...')
    
    # Stage 4: 分析生成
    print('\n--- Stage 4: 案例分析生成 ---')
    cases_text = json.dumps(
        [r.get('payload', {}) for r in top_results],
        ensure_ascii=False, indent=2,
    )
    elements_text = json.dumps(elements, ensure_ascii=False, indent=2)
    prompt = format_case_analysis_prompt(
        case_description=case_description,
        case_elements=elements_text,
        retrieved_cases=cases_text,
        n_cases=len(top_results),
    )
    analysis = llm_generate(prompt, temperature=0.3, max_tokens=4096)
    
    print('\n' + '=' * 60)
    print('分析报告')
    print('=' * 60)
    print(analysis)
    
    return {'elements': elements, 'retrieved_cases': top_results, 'analysis': analysis}

## Cell 10: 测试查询

In [ ]:
# 测试 1: 盗窃案
result1 = run_pipeline('被告人张某秘密窃取他人财物，价值人民币5000元')

In [ ]:
# 测试 2: 故意伤害案
result2 = run_pipeline('被告人李某因纠纷殴打他人，致人轻伤一级')

In [ ]:
# 测试 3: 诈骗案
result3 = run_pipeline('被告人赵某以虚假投资为名骗取他人钱财20万元')

## 说明

### 关于项目代码上传到 Kaggle

上面的 Notebook 假设 `src/` 目录存在。你需要把项目代码弄到 Kaggle 上：

**最简单的方式**：
1. 在本地把项目打包（排除 data/、models/、venv/、qdrant_data/）
2. 在 Kaggle 创建一个 Dataset，上传 zip
3. 在 Notebook 右侧 Add Input 选这个 Dataset
4. 在 Cell 2 里解压到 PROJECT_DIR

**或者**：如果项目在 GitHub 上，直接在 Cell 2 用 `!git clone` 拉取。

### 关于显存

T4 16GB 显存分配：
- BGE-M3: 约 2.3GB（向量化完成后释放）
- LawLLM-7B FP16: 约 14GB
- 推理时峰值: 约 15GB

如果显存不够，可以：
- 使用 AWQ 量化版（约 5-6GB）
- 减小 max_model_len（如 2048）
- 降低 gpu_memory_utilization（如 0.7）

### 关于 BGE-reranker

为了节省显存，Notebook 中跳过了 BGE-reranker，只用要素加权精排。
等流程跑通后，如果需要更好的重排序效果，可以再加载 BGE-reranker。